# Seaquest Stage-G0 — Full-View Four-Frame Critic (training half)
Train a FRESH full-view (oxygen mask OFF) four-frame contrastive critic with the FROZEN
pipeline — the ONLY change vs the masked four-frame critic is `oracle=True`. Gates: config
drift → 12-point implementation audit + visual grid → train (50k) → action-use sanity.
STOPS before closed-loop rollout (that runs locally on ALE/OCAtari). **Use:** TOKEN + Drive `raw_hf.zip`.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')


In [ ]:
# 1. Clone repo at the reviewed commit.
TOKEN = 'PASTE_YOUR_GITHUB_TOKEN_HERE'
import os, subprocess
D = '/content/Goal-Conditioned-Confounded-MsPacman'
if not os.path.isdir(D):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git',D], check=True)
%cd /content/Goal-Conditioned-Confounded-MsPacman
!git pull -q && git log --oneline -1


In [ ]:
# 2. Load raw_hf from Drive (mount + unzip + auto-locate traj_*.npz).
import glob, os, zipfile
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/raw_hf.zip'     # <-- EDIT if elsewhere
assert os.path.exists(ZIP), f'zip not found at {ZIP}'
EXTRACT = '/content/raw_hf_extracted'
if not glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True):
    with zipfile.ZipFile(ZIP) as z: z.extractall(EXTRACT)
DATA_ROOT = os.path.dirname(glob.glob(EXTRACT + '/**/traj_0000.npz', recursive=True)[0])
n = len(glob.glob(DATA_ROOT + '/traj_*.npz'))
print('DATA_ROOT =', DATA_ROOT, '| trajectories:', n); assert n == 40


In [ ]:
# 3. Step 4 — config diff vs masked critic (FULL_VIEW_CONFIG_DRIFT gate).
!python -m seaquest_ccrl.scripts.g0_config_diff


In [ ]:
# 4. Step 5 — implementation audit (12 assertions on real full-view batches) + visual grid.
!python -m seaquest_ccrl.scripts.g0_train_audit --root "$DATA_ROOT"
from IPython.display import Image, display
display(Image("artifacts/seaquest/goal_control/full_view/stack_visual_audit.png"))


In [ ]:
# 5. Steps 6-7 — train fresh full-view critic (oracle=True, 50k) + action-use sanity gate.
!python -m seaquest_ccrl.scripts.g0_train_fullview --root "$DATA_ROOT" --steps 50000
import json; print(json.dumps(json.load(open('artifacts/seaquest/goal_control/full_view/action_use_gate.json')), indent=2))


In [ ]:
# 6. Persist + download the training artifacts (checkpoint, history, audits, diagnostics).
import shutil
shutil.make_archive('seaquest_stage_g0_fullview_train', 'zip', 'artifacts/seaquest/goal_control/full_view')
try:
    from google.colab import drive; drive.mount('/content/drive')
    shutil.copy('seaquest_stage_g0_fullview_train.zip', '/content/drive/MyDrive'); print('copied to Drive')
except Exception as e: print('Drive optional:', e)
try:
    from google.colab import files; files.download('seaquest_stage_g0_fullview_train.zip')
except Exception: pass


## STOP — training half only
Send `action_use_gate.json` + `implementation_audit.json` + `config_diff_vs_masked.json`.
Closed-loop goal-reaching evaluation (the PRIMARY Stage-G0 gate) runs in the validated local
ALE/OCAtari environment with clone/restore — a separate stage, built after this passes review.
